# PCA analyses for ancestry filtering

## 1000 Genomes whole cohort PCA
Round 1: whole-genome PCA on all 1000G populations, HapMap3 SNPs, filtered on MAF, HWE, missingness, LD.

## Setup

plink2: manual install, not conda. Bucket: mounted via `gcsfuse` so plink reads/writes it as a local path.

In [ ]:
%%bash
set -e

# plink2
BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version

# bucket mount
MOUNT_DIR="$HOME/bucket"
mkdir -p "$MOUNT_DIR"
if command -v gcsfuse >/dev/null 2>&1; then
  mountpoint -q "$MOUNT_DIR" || gcsfuse --implicit-dirs "$(echo "$WORKSPACE_BUCKET" | sed 's#^gs://##')" "$MOUNT_DIR"
else
  echo "gcsfuse not found — confirm it's installed, or switch to gsutil cp/rsync" >&2
fi

In [ ]:
import os

# %%bash cells don't inherit PATH changes from prior cells — set it here instead
bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Paths

`BUCKET_DIR`: gcsfuse-mounted `$WORKSPACE_BUCKET/data/01_ancestry_filtering/`. See README's Layout convention for the full bucket structure.

In [ ]:
import os

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]  # confirm var name on Workbench 2.0

MOUNT_DIR = os.path.expanduser("~/bucket")
BUCKET_DIR = f"{MOUNT_DIR}/data/01_ancestry_filtering"
os.makedirs(BUCKET_DIR, exist_ok=True)

BFILE = f"{BUCKET_DIR}/1kg_all_chrs"                # genome-wide bed/bim/fam prefix
HM3_SNPLIST = f"{BUCKET_DIR}/hm3.snplist"
PANEL = f"{BUCKET_DIR}/integrated_call_samples_v3.20130502.ALL.panel"

OUT_PREFIX = f"{BUCKET_DIR}/1kg_all_hm3"            # post-QC bed/bim/fam
PRUNE_PREFIX = f"{BUCKET_DIR}/all_pruned"
PCA_PREFIX = f"{BUCKET_DIR}/round1_pca/1kg_all_pca"
os.makedirs(f"{BUCKET_DIR}/round1_pca", exist_ok=True)

## Build reference files

Prebuilt 1000G plink files ([plink2 resources](https://www.cog-genomics.org/plink/2.0/resources): 3,202 samples, GRCh38, pgen/pvar/psam) — no VCF download/merge needed. Restricted to HM3 SNPs and the classic 2,504 unrelated samples (via `PANEL`, same file round 2 uses) to stay consistent with round 2's ellipsoid fit.

In [ ]:
%%bash -s "$BUCKET_DIR" "$BFILE" "$HM3_SNPLIST" "$PANEL"
BUCKET_DIR=$1
BFILE=$2
HM3_SNPLIST=$3
PANEL=$4

cd "$BUCKET_DIR"

command -v zstd >/dev/null 2>&1 || {
  echo "zstd not found — install it first (e.g. 'sudo apt-get install -y zstd')" >&2
  exit 1
}

# HapMap3 SNP list
if [ ! -f "$(basename "$HM3_SNPLIST")" ]; then
  HM3_RAW=w_hm3.snplist
  if [ ! -f "$HM3_RAW" ]; then
    wget -q -O "${HM3_RAW}.bz2" "https://data.broadinstitute.org/alkesgroup/LDSCORE/w_hm3.snplist.bz2"
    bunzip2 -f "${HM3_RAW}.bz2"
  fi
  tail -n +2 "$HM3_RAW" | cut -f1 > "$(basename "$HM3_SNPLIST")"
fi

# classic 2,504 sample panel
if [ ! -f "$(basename "$PANEL")" ]; then
  wget -q -O "$(basename "$PANEL")" "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/integrated_call_samples_v3.20130502.ALL.panel"
fi
awk 'NR>1 {print $1}' "$(basename "$PANEL")" > classic_2504.keep

# prebuilt 1000G plink files (3,202 samples, GRCh38)
[ -f all_hg38.pgen ] || {
  wget -q -O all_hg38.pgen.zst "https://www.dropbox.com/s/j72j6uciq5zuzii/all_hg38.pgen.zst?dl=1"
  zstd -d --rm all_hg38.pgen.zst
}
[ -f all_hg38_rs.pvar.zst ] || wget -q -O all_hg38_rs.pvar.zst "https://www.dropbox.com/scl/fi/fn0bcm5oseyuawxfvkcpb/all_hg38_rs.pvar.zst"
[ -f hg38_corrected.psam ]  || wget -q -O hg38_corrected.psam  "https://www.dropbox.com/scl/fi/u5udzzaibgyvxzfnjcvjc/hg38_corrected.psam"

if [ ! -f "$(basename "$BFILE").bed" ]; then
  # restrict to classic samples + HM3 SNPs by native rsID
  plink2 \
    --pgen all_hg38.pgen \
    --pvar vzs all_hg38_rs.pvar.zst \
    --psam hg38_corrected.psam \
    --keep classic_2504.keep \
    --max-alleles 2 \
    --rm-dup exclude-all \
    --extract "$(basename "$HM3_SNPLIST")" \
    --make-bed \
    --out 1kg_hm3_raw

  # relabel IDs to chr:pos:ref:alt for matching against ACAF later
  plink2 \
    --bfile 1kg_hm3_raw \
    --set-all-var-ids '@:#:$r:$a' \
    --new-id-max-allele-len 1000 \
    --make-bed \
    --out "$(basename "$BFILE")"

  rm -f 1kg_hm3_raw.{bed,bim,fam,log}
fi

wc -l "$(basename "$BFILE").bim" "$(basename "$HM3_SNPLIST")" "$(basename "$BFILE").fam" classic_2504.keep

## Variant QC + LD pruning

No `--keep` (unlike round 2) — all populations, already HM3-restricted.

In [ ]:
%%bash -s "$BFILE" "$OUT_PREFIX" "$PRUNE_PREFIX"
BFILE=$1
OUT_PREFIX=$2
PRUNE_PREFIX=$3

plink2 \
  --bfile "$BFILE" \
  --maf 0.01 \
  --hwe 1e-5 \
  --geno 0.05 \
  --max-alleles 2 \
  --rm-dup exclude-all \
  --make-bed \
  --out "$OUT_PREFIX"

plink2 \
  --bfile "$OUT_PREFIX" \
  --indep-pairwise 200kb 1 0.1 \
  --out "$PRUNE_PREFIX"

wc -l "${PRUNE_PREFIX}.prune.in"

## PCA

20 PCs, `allele-wts` — loadings reused below to project AoU onto this space.

In [ ]:
%%bash -s "$OUT_PREFIX" "$PRUNE_PREFIX" "$PCA_PREFIX"
OUT_PREFIX=$1
PRUNE_PREFIX=$2
PCA_PREFIX=$3

plink2 \
  --bfile "$OUT_PREFIX" \
  --extract "${PRUNE_PREFIX}.prune.in" \
  --freq counts \
  --pca allele-wts 20 \
  --out "$PCA_PREFIX"

ls -lh "${PCA_PREFIX}".*

## Project AoU data onto the 1000G PCs

**File/format:** AoU ACAF threshold callset (GRCh38 srWGS, AF-filtered, chromosome-split pgen/pvar/psam).

**Variant filtering:** intersect, not re-filter — `--extract` the reference's already-pruned SNP list (`${PRUNE_PREFIX}.prune.in`) after re-IDing ACAF to the same `chr:pos:ref:alt` scheme. Overlap % reported as a sanity check.

**Mechanics:** `plink2 --score` against `.eigenvec.allele` + `.acount` from the PCA cell.

**Unconfirmed:** exact env var for the ACAF path (varies by CDR version). Next cell scans `os.environ` for it.

In [ ]:
import os

{k: v for k, v in os.environ.items() if "ACAF" in k.upper()}

In [ ]:
# PLACEHOLDER: unconfirmed var name. Replace with the actual key from the scan above.
ACAF_PGEN_PATTERN = os.environ["WGS_ACAF_THRESHOLD_SPLIT_PGEN_PATH"] + "/acaf_threshold.chr%d"

PROJECT_PREFIX = f"{BUCKET_DIR}/round1_pca/acaf_hm3_subset"
PROJECTED_OUT = f"{BUCKET_DIR}/round1_pca/acaf_projected"

### Extract + intersect

Per chromosome: re-ID ACAF to `chr:pos:ref:alt`, extract against `${PRUNE_PREFIX}.prune.in`, merge.

In [ ]:
%%bash -s "$ACAF_PGEN_PATTERN" "$PRUNE_PREFIX" "$PROJECT_PREFIX"
ACAF_PGEN_PATTERN=$1
PRUNE_PREFIX=$2
PROJECT_PREFIX=$3

WORK_DIR=$(dirname "$PROJECT_PREFIX")
cd "$WORK_DIR"

MERGE_LIST=acaf_merge_list.txt
> "$MERGE_LIST"

for CHR in $(seq 1 22); do
  CHR_PFILE=$(printf "$ACAF_PGEN_PATTERN" "$CHR")

  plink2 \
    --pfile "$CHR_PFILE" \
    --set-all-var-ids '@:#:$r:$a' \
    --new-id-max-allele-len 1000 \
    --extract "$PRUNE_PREFIX.prune.in" \
    --make-pgen \
    --out "acaf_chr${CHR}_subset"

  echo "acaf_chr${CHR}_subset" >> "$MERGE_LIST"
done

plink2 --pmerge-list "$MERGE_LIST" pfile --make-pgen --out "$(basename "$PROJECT_PREFIX")"

N_REF=$(wc -l < "${PRUNE_PREFIX}.prune.in")
N_OVERLAP=$(wc -l < "$(basename "$PROJECT_PREFIX").pvar")
echo "Reference pruned SNPs: $N_REF"
echo "ACAF overlap (incl. pvar header line): $N_OVERLAP"

### Project

`--score` against the reference's allele weights/counts. Column numbers read off the `.eigenvec.allele` header (layout varies by plink2 build).

In [ ]:
%%bash -s "$PROJECT_PREFIX" "$PCA_PREFIX" "$PROJECTED_OUT"
PROJECT_PREFIX=$1
PCA_PREFIX=$2
PROJECTED_OUT=$3

WEIGHTS="${PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")

ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

plink2 \
  --pfile "$PROJECT_PREFIX" \
  --read-freq "${PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$PROJECTED_OUT"

head "${PROJECTED_OUT}.sscore"